# Lab command cookbook

The commands of this lab that are hard to remember or to type: `kubectl` with jsonpath and jq,
`argocd`, `helm`, kro, ACK and Moto, Kyverno, Prometheus and Loki. Each section starts with the
concept in a few lines, then cells you can run as they are.

**Every cell only reads.** Running the whole notebook never changes the lab. Commands that change
something are shown as plain text in *Changes go through Git* at the end, with the Git-first way
to do the same thing. In this lab, Argo CD reverts most direct `kubectl` changes within minutes
(self-heal), so the durable change is always a commit.

**Setup (once):** `bash scripts/setup-notebook.sh`, then in VS Code
*Select Kernel → Jupyter Kernel… → Bash*. Details: [README.md](README.md).

**Secrets:** no cell prints a password, token or Secret value, and saved outputs are stripped
before a commit. Keep it that way when you add cells: `make password` belongs in the terminal.

Related reading: [Concepts and glossary](../../concepts-and-glossary.md),
[Lab 0 guided tour](../../lab-0-guided-tour.md), [runbooks](../../runbooks/).

## 0. Setup and tool check

Defines the variables the other cells use. Run it first, and again after restarting the kernel.

In [ ]:
HUB=k3d-hub-cluster
NONPROD=k3d-spoke-nonprod
PROD=k3d-spoke-prod
SPOKES=("$NONPROD" "$PROD")
SECRET_DIR="${GITOPS_LAB_SECRET_DIR:-$HOME/.config/gitops-lab}"
LAB_REPO="$(git rev-parse --show-toplevel 2>/dev/null || echo "$HOME/repos/gitops-control-plane")"
CATALOG_REPO="$(dirname "$LAB_REPO")/platform-catalog"
MOTO_ENDPOINT=http://localhost:5000

for t in kubectl argocd helm aws jq yq k3d docker; do
	if command -v "$t" >/dev/null; then echo "✔ $t"; else echo "✘ $t missing"; fi
done
for c in "$HUB" "$NONPROD" "$PROD"; do
	if kubectl config get-contexts -o name | grep -qx "$c"; then echo "✔ context $c"; else echo "✘ context $c missing"; fi
done
echo "lab repo: $LAB_REPO"
echo "catalog:  $CATALOG_REPO"

## 1. Where am I?

Three k3d clusters: the **hub** runs Argo CD, Keycloak, Grafana, Prometheus and Loki; the two
**spokes** run the platform add-ons (kro, ACK, Kyverno, …) and the tenant apps. Every command
names its cluster with `--context`, so it never depends on your current context.

In [ ]:
for c in "$HUB" "${SPOKES[@]}"; do
	echo "== $c"
	kubectl --context "$c" get nodes -o custom-columns='NODE:.metadata.name,READY:.status.conditions[?(@.type=="Ready")].status,VERSION:.status.nodeInfo.kubeletVersion,IP:.status.addresses[?(@.type=="InternalIP")].address'
done

Every Argo CD Application with its sync and health status, the one-line answer to *"is the lab
green?"*. Sync = does the cluster match Git; health = is what runs actually working.

In [ ]:
kubectl --context "$HUB" -n argocd get applications \
	-o custom-columns='APP:.metadata.name,PROJECT:.spec.project,SYNC:.status.sync.status,HEALTH:.status.health.status'

Only the problems (empty output means everything is Synced and Healthy):

In [ ]:
kubectl --context "$HUB" -n argocd get applications -o json | jq -r '
	.items[]
	| select(.status.sync.status != "Synced" or .status.health.status != "Healthy")
	| "\(.metadata.name)\t\(.status.sync.status)\t\(.status.health.status)\t\(.status.conditions[0].message // "")"'
echo "(end of list)"

## 2. Argo CD

- An **Application** points at one place in Git (a path, or a Helm chart plus values) and one
	destination cluster and namespace.
- An **ApplicationSet** generates Applications, here one per spoke from the cluster Secrets and
	their labels (`environment`) and annotations (`blueprints-revision`).
- An **AppProject** limits which repos, clusters, namespaces and kinds an Application may use.

The CLI below logs in as `platform-admin` with a **temporary config file**, so it never touches
your own `argocd` login. The password is read from its file and never shown.

In [ ]:
ARGOCD_CFG="$(mktemp)"
argocd login localhost --plaintext --grpc-web --skip-test-tls --config "$ARGOCD_CFG" \
	--username platform-admin --password "$(cat "$SECRET_DIR/argocd-platform-admin.password")" \
	</dev/null >/dev/null && echo "✔ logged in (temporary config, removed in the last cell)"
A=(argocd --config "$ARGOCD_CFG")

In [ ]:
"${A[@]}" app list

One app in detail: source, destination, conditions and every resource it manages.

In [ ]:
APP=orders-dev
"${A[@]}" app get "$APP"

Deployment history (which Git revision was synced when) and the live diff against Git. `app diff` exits with 1 when there is a difference, so the cell prints the code instead of failing.

In [ ]:
"${A[@]}" app history "$APP"
echo
"${A[@]}" app diff "$APP"; rc=$?
echo "app diff exit code: $rc (0 = live matches Git, 1 = differences shown above)"

Which ApplicationSet generated which Application:

In [ ]:
kubectl --context "$HUB" -n argocd get applications -o json | jq -r '
	.items[]
	| select(.metadata.ownerReferences[]?.kind == "ApplicationSet")
	| "\(.metadata.ownerReferences[0].name)\t\(.metadata.name)"' | sort | column -t

**Promotion state.** spoke-nonprod follows `main` of platform-catalog, spoke-prod a pinned tag.
The value lives on the cluster Secret (labels and annotations only are shown, never its data) and
comes from `clusters/blueprint-revisions.env` via `make promote-blueprints`.

In [ ]:
kubectl --context "$HUB" -n argocd get secrets -l argocd.argoproj.io/secret-type=cluster \
	-o custom-columns='CLUSTER:.metadata.name,ENV:.metadata.labels.environment,BLUEPRINTS:.metadata.annotations.blueprints-revision'
echo
grep -v '^#' "$LAB_REPO/clusters/blueprint-revisions.env"

## 3. Helm, and why `helm list` is (almost) empty

Argo CD renders Helm charts itself (`helm template`) and applies the result, so there are no Helm
*releases* for the charts it manages. On the spokes `helm list` shows only what k3s installs on
its own (Traefik). To see what Argo CD would apply, render the chart locally with the same values.

In [ ]:
helm --kube-context "$NONPROD" list -A

Every Helm chart Argo CD deploys, with its pinned version:

In [ ]:
kubectl --context "$HUB" -n argocd get applications -o json | jq -r '
	.items[] | .metadata.name as $app
	| (.spec.sources // [.spec.source])[]
	| select(.chart)
	| "\($app)\t\(.chart)\t\(.targetRevision)\t\(.repoURL)"' | sort | column -t

A chart's metadata and the first default values (HTTP repo and OCI registry):

In [ ]:
helm show chart kyverno --repo https://kyverno.github.io/kyverno/ --version 3.9.1 | grep --color=never -E '^(name|version|appVersion):'
echo
helm show chart oci://public.ecr.aws/aws-controllers-k8s/s3-chart --version 1.13.0 2>/dev/null | grep --color=never -E '^(name|version|appVersion):'
echo
helm show values kyverno --repo https://kyverno.github.io/kyverno/ --version 3.9.1 | head -20

Render Kyverno the way Argo CD does for the spokes: the catalog values file plus the parameter
from `applicationsets/addons-spoke-kyverno.yaml`, then list the kinds and names it would create.
The catalog checkout should match the spoke's revision (`main` for nonprod).

In [ ]:
helm template kyverno kyverno --repo https://kyverno.github.io/kyverno/ --version 3.9.1 -n kyverno \
	-f "$CATALOG_REPO/controllers/kyverno/values-kyverno.yaml" \
	--set reportsController.enabled=true \
	| yq -N '.kind + "/" + .metadata.name' | sort | uniq -c | sort -rn | head -25

## 4. kro: platform APIs

A **ResourceGraphDefinition** (RGD) defines a new Kubernetes kind (`QueueBackedService`,
`TestReportViewer`, …) and the resources one instance of it creates. A team writes a small
instance (a *claim*); kro creates and wires the children and reports their state back.

In [ ]:
kubectl --context "$NONPROD" get rgd

The fields a claim may set (kro generates the CRD, so `kubectl explain` works):

In [ ]:
kubectl --context "$NONPROD" explain queuebackedservice.spec

All instances on both spokes:

In [ ]:
for c in "${SPOKES[@]}"; do
	echo "== $c"
	kubectl --context "$c" get queuebackedservice,testreportviewer,teamekscluster -A 2>/dev/null || true
done

Everything one instance created, found through the `kro.run/instance-*` labels kro puts on its children:

In [ ]:
NS=orders-dev; NAME=orders
kubectl --context "$NONPROD" get deploy,svc,ingress,serviceaccount,configmap,queues.sqs.services.k8s.aws -n "$NS" \
	-l "kro.run/instance-name=$NAME" -o custom-columns='KIND:.kind,NAME:.metadata.name,NODE:.metadata.labels.kro\.run/node-id'

In [ ]:
kubectl --context "$NONPROD" -n "$NS" get queuebackedservice "$NAME" -o json \
	| jq -r '.status.conditions[]? | "\(.type)\t\(.status)\t\(.reason // "")\t\(.message // "")"' | column -t -s $'\t'

## 5. ACK and Moto: "AWS" resources

ACK controllers turn Kubernetes resources (`Queue`, `Bucket`, `Role`, `VPC`, …) into AWS API
calls. Here the AWS API is **Moto**, an in-memory emulator, so everything is local and lost when
Moto restarts (`make moto-restart` recreates it). Each namespace is mapped to an AWS account by
the annotation `services.k8s.aws/owner-account-id`.

In [ ]:
for c in "${SPOKES[@]}"; do
	echo "== $c"
	kubectl --context "$c" get ns -o custom-columns='NAMESPACE:.metadata.name,ACCOUNT:.metadata.annotations.services\.k8s\.aws/owner-account-id' \
		| awk 'NR==1 || $2!="<none>"'
done

ACK resources with their account and conditions (`ACK.ResourceSynced=True` means the AWS side matches):

In [ ]:
for c in "${SPOKES[@]}"; do
	echo "== $c"
	kubectl --context "$c" get queues.sqs.services.k8s.aws,buckets.s3.services.k8s.aws,roles.iam.services.k8s.aws -A -o json 2>/dev/null | jq -r '
		.items[]
		| "\(.kind)\t\(.metadata.namespace)/\(.metadata.name)\t\(.status.ackResourceMetadata.ownerAccountID // "-")\t\([.status.conditions[]? | "\(.type)=\(.status)"] | join(","))"' \
		| column -t -s $'\t'
done

Ask Moto directly, as one account. `moto_as` assumes the lab's `smoke-test` role in that account
and runs one `aws` command with the temporary credentials (they are never printed). Accounts:
`111111111111` spoke-nonprod, `222222222222` spoke-prod, `123456789012` Moto's default (should
stay empty). For the full inventory: `make list-moto-resources`.

In [ ]:
moto_as() {
	local account="$1"; shift
	local creds
	creds=$(AWS_ACCESS_KEY_ID=mock-key AWS_SECRET_ACCESS_KEY=mock-secret \
		aws --endpoint-url "$MOTO_ENDPOINT" --region us-east-1 --cli-connect-timeout 5 sts assume-role \
			--role-arn "arn:aws:iam::${account}:role/smoke-test" --role-session-name notebook \
			--query Credentials --output json) || return 1
	AWS_ACCESS_KEY_ID=$(jq -r .AccessKeyId <<<"$creds") AWS_SECRET_ACCESS_KEY=$(jq -r .SecretAccessKey <<<"$creds") \
		AWS_SESSION_TOKEN=$(jq -r .SessionToken <<<"$creds") \
		aws --endpoint-url "$MOTO_ENDPOINT" --region us-east-1 --cli-connect-timeout 5 "$@"
}
for acct in 111111111111 222222222222; do
	echo "== account $acct"
	moto_as "$acct" sqs list-queues --query 'QueueUrls' --output text
	moto_as "$acct" s3api list-buckets --query 'Buckets[].Name' --output text
done

## 6. Kyverno and Policy Reporter

Kyverno checks resources at **admission** (when they are created or changed) and in the
**background** (existing resources, written as PolicyReports). `Deny` policies block; `Audit`
policies only report. The UI for the reports is http://policy-reporter.localhost
(see [the Policy Reporter runbook](../../runbooks/policy-reporter.md)).

In [ ]:
kubectl --context "$NONPROD" get validatingpolicies.policies.kyverno.io,imagevalidatingpolicies.policies.kyverno.io -o json | jq -r '
	.items[]
	| "\(.metadata.name)\t\(.spec.validationActions // ["Deny"] | join(","))\t\(.metadata.annotations["policies.kyverno.io/severity"] // "-")\t\(.metadata.annotations["policies.kyverno.io/title"] // "-")"' \
	| column -t -s $'\t'

Pass/fail per policy on each spoke, from the PolicyReports:

In [ ]:
for c in "${SPOKES[@]}"; do
	echo "== $c"
	{ kubectl --context "$c" get policyreports.wgpolicyk8s.io -A -o json; kubectl --context "$c" get clusterpolicyreports.wgpolicyk8s.io -o json; } \
		| jq -rs '[.[].items[].results[]?] | group_by(.policy)[]
				| "\(.[0].policy)\tpass=\(map(select(.result=="pass"))|length)\tfail=\(map(select(.result=="fail"))|length)"' \
		| column -t -s $'\t'
done

Every failing resource, grouped by policy and namespace (the list to work through):

In [ ]:
kubectl --context "$NONPROD" get policyreports.wgpolicyk8s.io -A -o json | jq -r '
	.items[] | .metadata.namespace as $ns | .scope as $s
	| .results[]? | select(.result == "fail")
	| "\(.policy)\t\($ns)\t\($s.kind)/\($s.name)"' | sort | column -t -s $'\t'

**Why would this be denied?** `--dry-run=server` sends a request through all admission checks
(Pod Security, ValidatingAdmissionPolicies, Kyverno) without creating anything. The two requests
below go to the `admission-probes` namespace, which exists for exactly this. Both are expected to
be denied: the first by the registry allowlist (a ValidatingAdmissionPolicy), the second by the
Kyverno image policy `tenant-images-signed`. The cell prints each reason.

In [ ]:
POD_SPEC='{"spec":{"securityContext":{"runAsNonRoot":true,"runAsUser":1000,"seccompProfile":{"type":"RuntimeDefault"}},"containers":[{"name":"probe","image":"IMAGE","securityContext":{"allowPrivilegeEscalation":false,"capabilities":{"drop":["ALL"]}}}]}}'
try_pod() {
	kubectl --context "$NONPROD" -n admission-probes run notebook-probe --restart=Never --dry-run=server \
		--image="$1" --overrides="${POD_SPEC/IMAGE/$1}" 2>&1 | fold -s -w 140
	return 0
}
echo "# image from a registry outside the allowlist:"
try_pod docker.io/library/busybox:1.36
echo
echo "# tenant image Kyverno cannot verify (the tag does not exist, so there is no signed digest):"
try_pod ghcr.io/brunobml/orders-processor:not-signed

## 7. Observability: Prometheus and Loki

The spokes' Prometheus agents and Alloy send metrics and logs to the hub. Grafana
(http://grafana.localhost) is the place to look; these helpers query the same data from the
terminal by running `wget` inside the hub Prometheus pod, the same way the smoke tests do.

In [ ]:
promq() {
	kubectl --context "$HUB" -n monitoring exec deploy/prometheus-server -c prometheus-server -- \
		wget -qO- "http://localhost:9090/api/v1/query?query=$(jq -rn --arg q "$1" '$q|@uri')"
}
lokiq() {  # LogQL query over the last hour, newest first
	local q start
	q=$(jq -rn --arg q "$1" '$q|@uri')
	start=$(( $(date +%s) - 3600 ))000000000
	kubectl --context "$HUB" -n monitoring exec deploy/prometheus-server -c prometheus-server -- \
		wget -qO- "http://loki:3100/loki/api/v1/query_range?query=${q}&start=${start}&limit=${2:-10}"
}
echo "✔ promq and lokiq defined"

Targets per cluster that are up or down:

In [ ]:
promq 'sum by (cluster) (up)' | jq -r '.data.result[] | "up\t\(.metric.cluster // "(no cluster label)")\t\(.value[1])"'
promq 'up == 0' | jq -r '.data.result[] | "DOWN\t\(.metric.cluster // "hub")\t\(.metric.job)\t\(.metric.instance)"'

Kyverno admission requests per spoke and decision over the last hour:

In [ ]:
promq 'sum by (cluster, request_allowed) (increase(kyverno_admission_requests_total[1h]))' \
	| jq -r '.data.result[] | "\(.metric.cluster)\tallowed=\(.metric.request_allowed)\t\(.value[1] | tonumber | round)"' | column -t -s $'\t'

Which labels Loki knows, and the last log lines of one namespace containing `error`:

In [ ]:
kubectl --context "$HUB" -n monitoring exec deploy/prometheus-server -c prometheus-server -- \
	wget -qO- http://loki:3100/loki/api/v1/labels | jq -M -c .data
lokiq '{namespace="kyverno"} |= "error"' 5 \
	| jq -r '.data.result[] | .stream.pod as $p | .values[] | "\($p)  \(.[1] | .[0:160])"' \
	| sed 's/\x1b\[[0-9;]*m//g'   # drop the colour codes some apps write into their logs
echo "(end of log lines)"

## 8. Troubleshooting one-liners

Pods that are not Running or Completed, on all clusters:

In [ ]:
for c in "$HUB" "${SPOKES[@]}"; do
	echo "== $c"
	kubectl --context "$c" get pods -A --field-selector=status.phase!=Running,status.phase!=Succeeded --no-headers 2>&1
done

Warning events, newest last:

In [ ]:
kubectl --context "$NONPROD" get events -A --field-selector type=Warning --sort-by=.lastTimestamp \
	-o custom-columns='LAST:.lastTimestamp,NS:.metadata.namespace,OBJECT:.involvedObject.kind,NAME:.involvedObject.name,REASON:.reason' | tail -15

Pods with restarts, most first, and the biggest memory users:

In [ ]:
kubectl --context "$NONPROD" get pods -A -o json | jq -r '
	.items[] | {ns: .metadata.namespace, pod: .metadata.name, r: ([.status.containerStatuses[]?.restartCount] | add // 0)}
	| select(.r > 0) | "\(.r)\t\(.ns)/\(.pod)"' | sort -rn | head -10
echo
kubectl --context "$NONPROD" top pods -A --sort-by=memory | head -8

Every container image in a namespace (handy for the image policies):

In [ ]:
kubectl --context "$NONPROD" -n orders-dev get pods \
	-o jsonpath='{range .items[*]}{.metadata.name}{"\t"}{range .spec.containers[*]}{.image}{" "}{end}{"\n"}{end}'

API server health of each cluster:

In [ ]:
for c in "$HUB" "${SPOKES[@]}"; do
	printf '%-20s ' "$c"; kubectl --context "$c" get --raw='/readyz' 2>&1; echo
done

**Stale load balancer.** After a host restart a spoke's k3d load balancer can keep old node IPs, and
every second request then returns 404. Six requests in a row must all return 200 (smoke Gate 9b).
The fix is in *Changes go through Git* below (`make start` does it).

In [ ]:
for pair in orders-dev:8081 orders-test:8081 orders-prod:8082; do
	IFS=: read -r ns port <<<"$pair"
	codes=""
	for i in 1 2 3 4 5 6; do codes+="$(curl -s -m 5 -o /dev/null -w '%{http_code}' "http://${ns}.localhost:${port}/") "; done
	echo "${ns}: ${codes}"
done

## Changes go through Git

These commands change the lab. They are here so you recognize them, not to run from this
notebook. Prefer the Git route; use the direct one only in a drill or an incident.

| Goal | Git-first way | Direct command (drills and incidents) |
| --- | --- | --- |
| Change a manifest or values | edit, `git commit`, `git push`; Argo CD syncs | `kubectl apply` (self-heal reverts it) |
| Sync now instead of waiting | push, then refresh in the UI | `argocd app sync <app>` |
| Promote blueprints to prod | tag platform-catalog, edit `clusters/blueprint-revisions.env`, `make promote-blueprints` | — |
| Restart a workload | change its template in Git (a checksum annotation) | `kubectl rollout restart deploy/<name> -n <ns>` |
| Recover Moto after a restart | `make moto-restart` | — |
| Fix a stale spoke load balancer | `make start` | `docker exec k3d-<spoke>-serverlb nginx -s reload` |
| Renew credentials and tokens | `make maintain` | — |

Passwords: `make password` in a terminal, not in a cell (outputs end up in files).

## Cleanup

Removes the temporary Argo CD config of section 2.

In [ ]:
rm -f "${ARGOCD_CFG:-}" && unset ARGOCD_CFG A && echo "✔ temporary Argo CD config removed"